# Mission 1 - 신고자 성별 분류

## 1. 실험 개요

본 실험의 목표는 119 응급 신고 음성에서 신고자의 성별을 M/F로 분류하는 것이다. Log-Mel CNN, MFCC + RandomForest, F0 ablation, WavLM-base-plus 1-chunk, late fusion, WavLM 3-chunk inference를 순서대로 비교했다.

## 2. 데이터셋 및 전처리

실제 학습 로그에서 서울 Training 29,200개와 Validation 3,640개의 WAV/JSON pair가 모두 정상적으로 매칭되었고 누락은 없었다. 학습에는 서울 Training 데이터만 사용했으며, Validation 데이터는 학습에 사용하지 않고 평가와 모델 비교에만 사용했다.

annotation에서는 startAt, endAt, Speaker 정보를 신고자 발화 구간 추출에만 사용했다. 신고자 발화를 시간 순서대로 연결해 caller waveform을 구성했으며, gender는 학습 target과 Validation 평가 정답으로만 사용했다. text, symptom, address 등 다른 annotation 정보는 모델 입력으로 사용하지 않았다.


## 3. Baseline 1 - Log-Mel CNN

신고자 waveform에서 Log-Mel feature를 만들고 CNN으로 성별을 분류했다. 아래 셀은 실제 학습 코드와 epoch별 Training/Validation 로그를 보존한다. 최고 Validation 결과는 Accuracy 0.9396, Macro F1 0.9393이다.


In [ ]:
%cd /content/drive/MyDrive/mission1

!python train_cnn.py \
    --data_root "/content/drive/MyDrive/data(inho)" \
    --ckpt_path "/content/drive/MyDrive/saved_models/mission1/mission1_cnn.pt" \
    --epochs 10 \
    --batch_size 64 \
    --num_workers 8

/content/drive/MyDrive/mission1
[Training] WAV 개수       : 29200
[Training] JSON 개수      : 29200
[Training] 매칭된 pair 개수: 29200
[Training] JSON 없는 WAV   : 0
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Training samples  : 29,200
Validation samples: 3,640
Device: cuda
Train 다수 클래스 baseline: F (0.5323)
Epoch 1/10: 100% 457/457 [41:38<00:00,  5.47s/it]
Validation: 100% 3640/3640 [36:57<00:00,  1.64it/s]
epoch=1 train_loss=0.5242 train_acc=0.7888 val_acc=0.8720 val_macro_f1=0.8678
Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_cnn.pt
Epoch 2/10: 100% 457/457 [22:28<00:00,  2.95s/it]
Validation: 100% 3640/3640 [10:54<00:00,  5.56it/s]
epoch=2 train_loss=0.2938 train_acc=0.8834 val_acc=0.9137 val_macro_f1=0.9130
Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_cnn.pt
Epoch 3/10: 100% 457/457 [22:28<00:00,  2.95s/it]
Validation: 100% 3640/3640 [12:05<00:00,  5.0

## 4. Baseline 2 - MFCC + RandomForest

신고자 waveform에서 MFCC와 시간 통계 feature를 구성해 RandomForest로 분류했다. F0를 포함한 설정의 실제 Validation 결과는 Accuracy 0.9423, Macro F1 0.9419이다.


In [ ]:
#F0
%cd /content/drive/MyDrive/mission1

!python train_mfcc_rf.py \
    --data_root "/content/drive/MyDrive/data(inho)" \
    --ckpt_path "/content/drive/MyDrive/saved_models/mission1/mission1_mfcc_rf.joblib"

/content/drive/MyDrive/mission1
[Training] WAV 개수       : 29200
[Training] JSON 개수      : 29200
[Training] 매칭된 pair 개수: 29200
[Training] JSON 없는 WAV   : 0
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Training pairs  : 29,200
Validation pairs: 3,640
Train feature extraction: 100% 29200/29200 [5:42:46<00:00,  1.42it/s]
Val feature extraction: 100% 3640/3640 [38:59<00:00,  1.56it/s]
Train feature shape: (29200, 246)
Val feature shape  : (3640, 246)
F0 사용: True
RandomForest 학습 시작...
[참고] Train accuracy: 1.0000
Validation accuracy : 0.9423
Validation macro F1 : 0.9419

Validation classification report
              precision    recall  f1-score   support

           M     0.9428    0.9316    0.9372      1681
           F     0.9419    0.9515    0.9467      1959

    accuracy                         0.9423      3640
   macro avg     0.9423    0.9415    0.9419      3640
weighted avg     0.9423    0.9423    0.94

### F0 Ablation

F0 feature의 영향을 확인하기 위해 동일한 MFCC + RandomForest 설정에서 F0를 제외했다. F0 포함 설정의 Accuracy는 0.9423, F0 제외 설정의 Accuracy는 0.9365로, 이 Validation 실험에서는 F0를 추가했을 때 Accuracy가 0.58%p 높았다.


In [ ]:
!python train_mfcc_rf.py \
    --data_root "/content/drive/MyDrive/data(inho)" \
    --no_f0 \
    --ckpt_path "/content/drive/MyDrive/saved_models/mission1/mission1_mfcc_rf_no_f0.joblib"

[Training] WAV 개수       : 29200
[Training] JSON 개수      : 29200
[Training] 매칭된 pair 개수: 29200
[Training] JSON 없는 WAV   : 0
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Training pairs  : 29,200
Validation pairs: 3,640
Train feature extraction: 100% 29200/29200 [44:09<00:00, 11.02it/s]
Val feature extraction: 100% 3640/3640 [05:29<00:00, 11.05it/s]
Train feature shape: (29200, 240)
Val feature shape  : (3640, 240)
F0 사용: False
RandomForest 학습 시작...
[참고] Train accuracy: 1.0000
Validation accuracy : 0.9365
Validation macro F1 : 0.9361

Validation classification report
              precision    recall  f1-score   support

           M     0.9394    0.9221    0.9307      1681
           F     0.9342    0.9490    0.9415      1959

    accuracy                         0.9365      3640
   macro avg     0.9368    0.9355    0.9361      3640
weighted avg     0.9366    0.9365    0.9365      3640

Train 다수 클래스 baseli

## 5. Proposed Model - WavLM-base-plus 1-Chunk

사전학습 encoder로 microsoft/wavlm-base-plus를 사용했다. WavLM encoder는 고정(frozen)한 상태에서 embedding을 추출했으며, annotation의 startAt, endAt, Speaker 정보를 이용해 신고자 발화 구간만 추출하고 해당 waveform을 encoder에 입력했다. 마지막 hidden representation에 mean pooling을 적용해 768차원 embedding을 생성했다.

Training 29,200개와 Validation 3,640개의 embedding을 같은 방식으로 cache에 저장했다. Validation 3,640개 전체의 embedding을 cache로 저장했으며, 이후 학습 셀에서 `(3640, 768)` shape의 cache를 다시 로드하여 정상 생성 여부를 확인했다. 아래 첫 셀은 Validation embedding 생성 과정을, 이어지는 셀은 Training/Validation embedding cache를 불러와 classification head를 학습하는 과정을 보여준다.

Embedding 추출 뒤에는 StandardScaler와 MLP classification head를 사용했다. MLP는 Linear(768 → 256), ReLU, Dropout(0.2), Linear(256 → 2)로 구성했다. 실제 head 학습의 최고 Validation 결과는 epoch 9에서 Accuracy 0.9566, Macro F1 0.9563이다.


In [ ]:
%cd /content/drive/MyDrive/mission1

import torch
import numpy as np
from pathlib import Path
from transformers import AutoFeatureExtractor, AutoModel

from train_wavlm import collect_split_pairs, extract_embeddings

MODEL_NAME = "microsoft/wavlm-base-plus"

DATA_ROOT = Path("/content/drive/MyDrive/data(inho)")
CACHE_DIR = Path("/content/drive/MyDrive/mission1/wavlm_embedding_cache")

VAL_CACHE = CACHE_DIR / "val_microsoft_wavlm-base-plus_8s_1chunks.npz"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# Validation만 탐색
print("\nValidation 데이터 찾는 중...")
val_samples = collect_split_pairs(DATA_ROOT, "Validation")
print(f"Validation samples: {len(val_samples):,}")

# 혹시 이미 validation cache가 있으면 재추출하지 않음
if VAL_CACHE.exists():
    print("\n✅ Validation cache가 이미 존재합니다.")
    print(VAL_CACHE)

else:
    print("\nWavLM 로드 중...")

    feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_NAME)
    encoder = AutoModel.from_pretrained(MODEL_NAME).to(device)
    encoder.eval()

    for p in encoder.parameters():
        p.requires_grad = False

    print("\n=== Validation WavLM embedding 추출 ===")

    x_val, y_val = extract_embeddings(
        val_samples,
        feature_extractor=feature_extractor,
        encoder=encoder,
        device=device,
        max_seconds=8,
        chunks_per_call=1,
        timestamp_unit="auto",
        desc="Val WavLM embedding",
    )

    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        VAL_CACHE,
        x=x_val,
        y=y_val,
    )

    print("\n✅ Validation embedding 완료!")
    print("shape:", x_val.shape)
    print("labels:", y_val.shape)
    print("저장 위치:", VAL_CACHE)

/content/drive/MyDrive/mission1
Device: cpu

Validation 데이터 찾는 중...
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0
Validation samples: 3,640

WavLM 로드 중...


preprocessor_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/2.23k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  378MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  378MB            

... (2 outputs left out)


=== Validation WavLM embedding 추출 ===


Val WavLM embedding:   0%|          | 0/3640 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(



✅ Validation embedding 완료!
shape: (3640, 768)
labels: (3640,)
저장 위치: /content/drive/MyDrive/mission1/wavlm_embedding_cache/val_microsoft_wavlm-base-plus_8s_1chunks.npz


In [ ]:
%cd /content/drive/MyDrive/mission1

import joblib
import numpy as np
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.preprocessing import StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


# =========================
# 설정
# =========================

CACHE_DIR = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache"
)

TRAIN_CACHE = CACHE_DIR / (
    "train_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

VAL_CACHE = CACHE_DIR / (
    "val_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

CKPT_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

SCALER_PATH = CKPT_PATH.with_suffix(".scaler.joblib")

EPOCHS = 15
BATCH_SIZE = 256
LEARNING_RATE = 1e-3
HIDDEN_DIM = 256
DROPOUT = 0.2
SEED = 42


# =========================
# seed / device
# =========================

np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)


# =========================
# 캐시 확인
# =========================

assert TRAIN_CACHE.exists(), (
    f"Train cache가 없습니다: {TRAIN_CACHE}"
)

assert VAL_CACHE.exists(), (
    f"Validation cache가 없습니다: {VAL_CACHE}"
)

print("✅ Train cache 확인:", TRAIN_CACHE)
print("✅ Val cache 확인  :", VAL_CACHE)


# =========================
# embedding 로드
# =========================

print("\nEmbedding cache 로드 중...")

train_data = np.load(TRAIN_CACHE)
val_data = np.load(VAL_CACHE)

x_train = train_data["x"].astype(np.float32)
y_train = train_data["y"].astype(np.int64)

x_val = val_data["x"].astype(np.float32)
y_val = val_data["y"].astype(np.int64)

print("Train embedding shape:", x_train.shape)
print("Train label shape    :", y_train.shape)

print("Val embedding shape  :", x_val.shape)
print("Val label shape      :", y_val.shape)


# =========================
# 데이터 sanity check
# =========================

print("\nTrain label distribution:")
unique, counts = np.unique(y_train, return_counts=True)

for u, c in zip(unique, counts):
    print(f"class {u}: {c}")

print("\nValidation label distribution:")
unique, counts = np.unique(y_val, return_counts=True)

for u, c in zip(unique, counts):
    print(f"class {u}: {c}")


# =========================
# StandardScaler
# =========================

scaler = StandardScaler()

x_train = scaler.fit_transform(
    x_train
).astype(np.float32)

x_val = scaler.transform(
    x_val
).astype(np.float32)


# =========================
# Dataset / DataLoader
# =========================

train_dataset = TensorDataset(
    torch.from_numpy(x_train),
    torch.from_numpy(y_train),
)

val_dataset = TensorDataset(
    torch.from_numpy(x_val),
    torch.from_numpy(y_val),
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available(),
)


# =========================
# Classification Head
# =========================

class Head(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=256,
        dropout=0.2,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )

    def forward(self, x):
        return self.net(x)


model = Head(
    input_dim=x_train.shape[1],
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
).to(device)


# =========================
# 평가 함수
# =========================

def evaluate(model, loader):

    model.eval()

    all_pred = []
    all_true = []

    with torch.inference_mode():

        for x, y in loader:

            x = x.to(
                device,
                non_blocking=True
            )

            logits = model(x)

            pred = (
                logits
                .argmax(dim=1)
                .cpu()
                .numpy()
            )

            all_pred.extend(pred.tolist())
            all_true.extend(y.numpy().tolist())

    acc = accuracy_score(
        all_true,
        all_pred
    )

    macro_f1 = f1_score(
        all_true,
        all_pred,
        average="macro"
    )

    return (
        acc,
        macro_f1,
        np.asarray(all_true),
        np.asarray(all_pred),
    )


# =========================
# 학습 준비
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

criterion = nn.CrossEntropyLoss()

best_f1 = -1.0
best_acc = -1.0
best_epoch = -1

CKPT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


# =========================
# 학습
# =========================

print(
    "\n=== Frozen WavLM embedding "
    "+ classification head 학습 ==="
)

for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, y in train_loader:

        x = x.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        logits = model(x)

        loss = criterion(
            logits,
            y
        )

        loss.backward()
        optimizer.step()

        running_loss += (
            loss.item()
            * y.size(0)
        )

        correct += (
            logits.argmax(dim=1)
            == y
        ).sum().item()

        total += y.size(0)

    train_loss = (
        running_loss
        / total
    )

    train_acc = (
        correct
        / total
    )

    (
        val_acc,
        val_f1,
        _,
        _
    ) = evaluate(
        model,
        val_loader
    )

    print(
        f"epoch={epoch:02d} "
        f"train_loss={train_loss:.4f} "
        f"train_acc={train_acc:.4f} "
        f"val_acc={val_acc:.4f} "
        f"val_macro_f1={val_f1:.4f}"
    )

    # best macro F1 기준 저장
    if val_f1 > best_f1:

        best_f1 = val_f1
        best_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "model_name":
                    "microsoft/wavlm-base-plus",

                "head_state_dict":
                    model.state_dict(),

                "input_dim":
                    x_train.shape[1],

                "hidden_dim":
                    HIDDEN_DIM,

                "dropout":
                    DROPOUT,

                "max_seconds":
                    8,

                "chunks_per_call":
                    1,

                "best_epoch":
                    best_epoch,

                "best_val_accuracy":
                    best_acc,

                "best_val_macro_f1":
                    best_f1,
            },
            CKPT_PATH,
        )

        joblib.dump(
            scaler,
            SCALER_PATH
        )

        print(
            "  ✅ Best checkpoint 저장:",
            CKPT_PATH
        )


# =========================
# BEST 모델 다시 로드
# =========================

checkpoint = torch.load(
    CKPT_PATH,
    map_location=device,
    weights_only=False,
)

model.load_state_dict(
    checkpoint["head_state_dict"]
)


# =========================
# 최종 Validation 평가
# =========================

(
    final_acc,
    final_f1,
    y_true,
    y_pred,
) = evaluate(
    model,
    val_loader
)

print("\n============================")
print(" BEST VALIDATION RESULT")
print("============================")

print(
    "Best epoch          :",
    checkpoint["best_epoch"]
)

print(
    f"Validation accuracy : "
    f"{final_acc:.4f}"
)

print(
    f"Validation macro F1 : "
    f"{final_f1:.4f}"
)

print("\nClassification Report")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "class 0",
            "class 1"
        ],
        digits=4,
    )
)

print("\n✅ checkpoint:")
print(CKPT_PATH)

print("\n✅ scaler:")
print(SCALER_PATH)

/content/drive/MyDrive/mission1
Device: cpu
✅ Train cache 확인: /content/drive/MyDrive/mission1/wavlm_embedding_cache/train_microsoft_wavlm-base-plus_8s_1chunks.npz
✅ Val cache 확인  : /content/drive/MyDrive/mission1/wavlm_embedding_cache/val_microsoft_wavlm-base-plus_8s_1chunks.npz

Embedding cache 로드 중...
Train embedding shape: (29200, 768)
Train label shape    : (29200,)
Val embedding shape  : (3640, 768)
Val label shape      : (3640,)

Train label distribution:
class 0: 13658
class 1: 15542

Validation label distribution:
class 0: 1681
class 1: 1959

=== Frozen WavLM embedding + classification head 학습 ===
epoch=01 train_loss=0.2070 train_acc=0.9212 val_acc=0.9437 val_macro_f1=0.9433
  ✅ Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_wavlm_1chunk.pt
epoch=02 train_loss=0.1509 train_acc=0.9465 val_acc=0.9495 val_macro_f1=0.9492
  ✅ Best checkpoint 저장: /content/drive/MyDrive/saved_models/mission1/mission1_wavlm_1chunk.pt
epoch=03 train_loss=0.1389 train_acc=0.95

## 6. WavLM 1-Chunk + RF Late Fusion

WavLM 1-chunk와 MFCC+F0 RandomForest의 class probability를 가중 평균하는 late fusion을 비교했다. 실제 weight sweep에서 WavLM 0.45, RF 0.55일 때 Accuracy 0.9635, Macro F1 0.9632를 기록했다. 이 weight는 동일한 Validation 결과를 기준으로 선택한 값이다.


In [ ]:
%cd /content/drive/MyDrive/mission1

import joblib
import numpy as np
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

from train_mfcc_rf import collect_split_pairs, build_features
from train_wavlm import Head
from common import INDEX_TO_GENDER


# =========================================================
# 경로 설정
# =========================================================

DATA_ROOT = Path("/content/drive/MyDrive/data(inho)")

RF_CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_mfcc_rf.joblib"
)

WAVLM_VAL_CACHE = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

WAVLM_CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

WAVLM_SCALER = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.scaler.joblib"
)


# =========================================================
# 1. RandomForest 모델 로드
# =========================================================

rf_data = joblib.load(RF_CKPT)

rf_model = rf_data["classifier"]
n_mfcc = rf_data["n_mfcc"]
use_f0 = rf_data["use_f0"]
timestamp_unit = rf_data["timestamp_unit"]

print("=== RF 모델 설정 ===")
print("n_mfcc        :", n_mfcc)
print("use_f0        :", use_f0)
print("timestamp_unit:", timestamp_unit)
print("classes       :", rf_model.classes_)


# =========================================================
# 2. Validation MFCC + F0 feature 추출
# =========================================================

print("\n=== Validation MFCC+F0 feature 추출 ===")

val_pairs = collect_split_pairs(
    DATA_ROOT,
    "Validation"
)

x_val_rf, y_val_rf = build_features(
    val_pairs,
    n_mfcc=n_mfcc,
    use_f0=use_f0,
    timestamp_unit=timestamp_unit,
    desc="Validation RF feature",
)

print("\nRF feature shape:", x_val_rf.shape)
print("RF label shape  :", y_val_rf.shape)


# =========================================================
# 3. RF 확률 예측
# =========================================================

rf_prob = rf_model.predict_proba(x_val_rf)

print("RF probability shape:", rf_prob.shape)


# =========================================================
# 4. WavLM Validation embedding 로드
# =========================================================

wavlm_data = np.load(WAVLM_VAL_CACHE)

x_val_wavlm = wavlm_data["x"].astype(np.float32)
y_val_wavlm = wavlm_data["y"].astype(np.int64)

print("\nWavLM embedding shape:", x_val_wavlm.shape)
print("WavLM label shape    :", y_val_wavlm.shape)


# =========================================================
# 5. Validation 순서 일치 확인
# =========================================================

assert len(y_val_rf) == len(y_val_wavlm), (
    "RF와 WavLM Validation 샘플 수가 다릅니다."
)

assert np.array_equal(y_val_rf, y_val_wavlm), (
    "RF와 WavLM Validation 순서가 다릅니다."
)

print("\n✅ RF / WavLM Validation 순서 일치")


# =========================================================
# 6. WavLM scaler 적용
# =========================================================

scaler = joblib.load(WAVLM_SCALER)

x_val_scaled = scaler.transform(
    x_val_wavlm
).astype(np.float32)


# =========================================================
# 7. WavLM classification head 로드
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

checkpoint = torch.load(
    WAVLM_CKPT,
    map_location=device,
    weights_only=False,
)

wavlm_model = Head(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    dropout=checkpoint["dropout"],
).to(device)

wavlm_model.load_state_dict(
    checkpoint["head_state_dict"]
)

wavlm_model.eval()


# =========================================================
# 8. WavLM 확률 예측
# =========================================================

wavlm_probs = []

batch_size = 512

with torch.inference_mode():

    for start in range(
        0,
        len(x_val_scaled),
        batch_size
    ):

        x = torch.from_numpy(
            x_val_scaled[start:start + batch_size]
        ).to(device)

        logits = wavlm_model(x)

        prob = torch.softmax(
            logits,
            dim=1
        )

        wavlm_probs.append(
            prob.cpu().numpy()
        )

wavlm_prob = np.vstack(wavlm_probs)

print("WavLM probability shape:", wavlm_prob.shape)


# =========================================================
# 9. 단독 모델 성능 확인
# =========================================================

y_true = y_val_wavlm

rf_pred = rf_prob.argmax(axis=1)
wavlm_pred = wavlm_prob.argmax(axis=1)

rf_acc = accuracy_score(
    y_true,
    rf_pred
)

rf_f1 = f1_score(
    y_true,
    rf_pred,
    average="macro"
)

wavlm_acc = accuracy_score(
    y_true,
    wavlm_pred
)

wavlm_f1 = f1_score(
    y_true,
    wavlm_pred,
    average="macro"
)

print("\n============================")
print(" 단독 모델 성능")
print("============================")

print(
    f"RF     | acc={rf_acc:.4f} "
    f"| macro_f1={rf_f1:.4f}"
)

print(
    f"WavLM  | acc={wavlm_acc:.4f} "
    f"| macro_f1={wavlm_f1:.4f}"
)


# =========================================================
# 10. Late Fusion
# =========================================================

print("\n============================")
print(" Late Fusion")
print("============================")

results = []

for wavlm_weight in np.arange(
    0.0,
    1.01,
    0.05
):

    rf_weight = 1.0 - wavlm_weight

    fusion_prob = (
        wavlm_weight * wavlm_prob
        + rf_weight * rf_prob
    )

    fusion_pred = fusion_prob.argmax(
        axis=1
    )

    acc = accuracy_score(
        y_true,
        fusion_pred
    )

    macro_f1 = f1_score(
        y_true,
        fusion_pred,
        average="macro"
    )

    results.append(
        (
            wavlm_weight,
            rf_weight,
            acc,
            macro_f1,
        )
    )

    print(
        f"WavLM={wavlm_weight:.2f} "
        f"RF={rf_weight:.2f} "
        f"| acc={acc:.4f} "
        f"| macro_f1={macro_f1:.4f}"
    )


# =========================================================
# 11. 최고 Fusion 조합 찾기
# =========================================================

best = max(
    results,
    key=lambda x: x[3]
)

best_wavlm_weight = best[0]
best_rf_weight = best[1]
best_acc = best[2]
best_f1 = best[3]

print("\n============================")
print(" BEST FUSION")
print("============================")

print(
    f"WavLM weight : {best_wavlm_weight:.2f}"
)

print(
    f"RF weight    : {best_rf_weight:.2f}"
)

print(
    f"Accuracy     : {best_acc:.4f}"
)

print(
    f"Macro F1     : {best_f1:.4f}"
)


# =========================================================
# 12. 최고 Fusion classification report
# =========================================================

best_prob = (
    best_wavlm_weight * wavlm_prob
    + best_rf_weight * rf_prob
)

best_pred = best_prob.argmax(
    axis=1
)

target_names = [
    INDEX_TO_GENDER[i]
    for i in range(len(INDEX_TO_GENDER))
]

print("\n=== BEST FUSION Classification Report ===")

print(
    classification_report(
        y_true,
        best_pred,
        target_names=target_names,
        digits=4,
        zero_division=0,
    )
)

/content/drive/MyDrive/mission1
=== RF 모델 설정 ===
n_mfcc        : 20
use_f0        : True
timestamp_unit: auto
classes       : [0 1]

=== Validation MFCC+F0 feature 추출 ===
[Validation] WAV 개수       : 3640
[Validation] JSON 개수      : 3640
[Validation] 매칭된 pair 개수: 3640
[Validation] JSON 없는 WAV   : 0


Validation RF feature: 100%|██████████| 3640/3640 [14:11<00:00,  4.28it/s]



RF feature shape: (3640, 246)
RF label shape  : (3640,)
RF probability shape: (3640, 2)

WavLM embedding shape: (3640, 768)
WavLM label shape    : (3640,)

✅ RF / WavLM Validation 순서 일치
Device: cpu
WavLM probability shape: (3640, 2)

 단독 모델 성능
RF     | acc=0.9423 | macro_f1=0.9419
WavLM  | acc=0.9566 | macro_f1=0.9563

 Late Fusion
WavLM=0.00 RF=1.00 | acc=0.9423 | macro_f1=0.9419
WavLM=0.05 RF=0.95 | acc=0.9451 | macro_f1=0.9447
WavLM=0.10 RF=0.90 | acc=0.9478 | macro_f1=0.9475
WavLM=0.15 RF=0.85 | acc=0.9530 | macro_f1=0.9527
WavLM=0.20 RF=0.80 | acc=0.9547 | macro_f1=0.9544
WavLM=0.25 RF=0.75 | acc=0.9580 | macro_f1=0.9577
WavLM=0.30 RF=0.70 | acc=0.9596 | macro_f1=0.9593
WavLM=0.35 RF=0.65 | acc=0.9607 | macro_f1=0.9604
WavLM=0.40 RF=0.60 | acc=0.9624 | macro_f1=0.9621
WavLM=0.45 RF=0.55 | acc=0.9635 | macro_f1=0.9632
WavLM=0.50 RF=0.50 | acc=0.9624 | macro_f1=0.9621
WavLM=0.55 RF=0.45 | acc=0.9629 | macro_f1=0.9626
WavLM=0.60 RF=0.40 | acc=0.9629 | macro_f1=0.9626
WavLM=0.65 RF=0

## 7. WavLM 3-Chunk Inference

1-chunk 방식은 긴 신고자 음성에서 일부 구간만 사용한다는 한계가 있다. 이를 보완하기 위해 신고자 발화를 시간 순서대로 연결한 waveform이 8초보다 길면 front, middle, end의 최대 8초 구간을 각각 사용했다. 각 chunk를 같은 WavLM encoder로 처리한 embedding의 평균을 통화 representation으로 사용했으며, 8초 이하 waveform은 zero-padding 후 한 번만 처리했다.

3-chunk용 classification head를 새로 학습한 것은 아니다. 1-chunk 실험에서 학습한 StandardScaler와 MLP head를 그대로 사용하고, Validation inference의 embedding aggregation만 3-chunk 평균으로 변경했다. 실제 비교 결과 1-chunk는 Accuracy 0.9566, Macro F1 0.9563이었고, 3-chunk는 Accuracy 0.9750, Macro F1 0.9748로 Accuracy가 1.84%p 높았다.


In [ ]:
%cd /content/drive/MyDrive/mission1

import numpy as np
import joblib
import torch

from pathlib import Path
from sklearn.metrics import accuracy_score, f1_score, classification_report

from train_wavlm import Head
from common import INDEX_TO_GENDER


# =========================================================
# 경로
# =========================================================

CACHE_1 = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_1chunks.npz"
)

CACHE_3 = Path(
    "/content/drive/MyDrive/mission1/wavlm_embedding_cache/"
    "val_microsoft_wavlm-base-plus_8s_3chunks.npz"
)

CKPT = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

SCALER_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.scaler.joblib"
)


# =========================================================
# 데이터 로드
# =========================================================

data1 = np.load(CACHE_1)
data3 = np.load(CACHE_3)

x1 = data1["x"].astype(np.float32)
y1 = data1["y"].astype(np.int64)

x3 = data3["x"].astype(np.float32)
y3 = data3["y"].astype(np.int64)

print("1-chunk:", x1.shape)
print("3-chunk:", x3.shape)

assert np.array_equal(y1, y3)

print("✅ label 순서 일치")


# =========================================================
# scaler
# =========================================================

scaler = joblib.load(SCALER_PATH)

x1_scaled = scaler.transform(x1).astype(np.float32)
x3_scaled = scaler.transform(x3).astype(np.float32)


# =========================================================
# WavLM head 로드
# =========================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

checkpoint = torch.load(
    CKPT,
    map_location=device,
    weights_only=False
)

model = Head(
    input_dim=checkpoint["input_dim"],
    hidden_dim=checkpoint["hidden_dim"],
    dropout=checkpoint["dropout"]
).to(device)

model.load_state_dict(
    checkpoint["head_state_dict"]
)

model.eval()


# =========================================================
# probability 함수
# =========================================================

def predict_prob(x):

    probs = []

    batch_size = 512

    with torch.inference_mode():

        for start in range(
            0,
            len(x),
            batch_size
        ):

            batch = torch.from_numpy(
                x[start:start + batch_size]
            ).to(device)

            logits = model(batch)

            prob = torch.softmax(
                logits,
                dim=1
            )

            probs.append(
                prob.cpu().numpy()
            )

    return np.vstack(probs)


# =========================================================
# 예측
# =========================================================

prob1 = predict_prob(x1_scaled)
prob3 = predict_prob(x3_scaled)

pred1 = prob1.argmax(axis=1)
pred3 = prob3.argmax(axis=1)


# =========================================================
# 성능
# =========================================================

acc1 = accuracy_score(y1, pred1)
f1_1 = f1_score(
    y1,
    pred1,
    average="macro"
)

acc3 = accuracy_score(y3, pred3)
f1_3 = f1_score(
    y3,
    pred3,
    average="macro"
)


print("\n============================")
print(" WavLM 비교")
print("============================")

print(
    f"1-chunk | "
    f"acc={acc1:.4f} | "
    f"macro_f1={f1_1:.4f}"
)

print(
    f"3-chunk | "
    f"acc={acc3:.4f} | "
    f"macro_f1={f1_3:.4f}"
)

print("\nAccuracy 변화:")
print(
    f"{acc1:.4f} → {acc3:.4f} "
    f"({(acc3-acc1)*100:+.2f}%p)"
)


print("\n=== 3-chunk Classification Report ===")

target_names = [
    INDEX_TO_GENDER[i]
    for i in range(len(INDEX_TO_GENDER))
]

print(
    classification_report(
        y3,
        pred3,
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)

/content/drive/MyDrive/mission1
1-chunk: (3640, 768)
3-chunk: (3640, 768)
✅ label 순서 일치

 WavLM 비교
1-chunk | acc=0.9566 | macro_f1=0.9563
3-chunk | acc=0.9750 | macro_f1=0.9748

Accuracy 변화:
0.9566 → 0.9750 (+1.84%p)

=== 3-chunk Classification Report ===
              precision    recall  f1-score   support

           M     0.9783    0.9673    0.9728      1681
           F     0.9722    0.9816    0.9769      1959

    accuracy                         0.9750      3640
   macro avg     0.9753    0.9745    0.9748      3640
weighted avg     0.9750    0.9750    0.9750      3640



## 8. WavLM 3-Chunk + RF Late Fusion

WavLM 3-chunk probability와 MFCC+F0 RandomForest probability를 결합하는 late fusion을 평가했다. 아래 셀은 3-chunk cache, 기존 WavLM head·scaler, 기존 RF checkpoint를 사용해 수행한 실제 재현 결과를 보존한다.

Validation 3,640개의 MFCC+F0 feature를 추출했으며, 추출 결과의 shape는 `(3640, 246)`이었다. 기존 RandomForest의 Validation Accuracy `0.9423`도 stored/current 결과에서 동일하게 재현되는 것을 확인했다.

WavLM 0.85와 RF 0.15의 가중 평균에서 Accuracy 0.9755, Macro F1 0.9754를 기록했다. WavLM 3-chunk 단독 대비 Accuracy 차이는 0.05%p이며, 예측 12개가 변경되어 오답→정답 7개, 정답→오답 5개, 순개선 2개였다. 이 weight는 동일한 Validation set의 성능을 기준으로 선택했으므로, 이 결과를 독립 test set에서 검증된 일반화 성능으로 해석하지 않는다.


In [2]:
# ============================================================
# Mission 1
# WavLM-base-plus 3-Chunk + MFCC/F0 RandomForest Late Fusion
#
# - WavLM encoder 재실행 X
# - Training feature 재추출 X
# - Validation 3,640개 RF feature만 추출
# - 기존 3-chunk WavLM cache 사용
# ============================================================

import sys
from pathlib import Path

import joblib
import numpy as np
import torch

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
)

from tqdm.auto import tqdm


# ============================================================
# 1. 경로
# ============================================================

MISSION_DIR = Path("/content/drive/MyDrive/mission1")
DATA_ROOT = Path("/content/drive/MyDrive/data(inho)")

WAVLM_CACHE_PATH = (
    MISSION_DIR
    / "wavlm_embedding_cache"
    / "val_microsoft_wavlm-base-plus_8s_3chunks.npz"
)

WAVLM_HEAD_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.pt"
)

WAVLM_SCALER_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_wavlm_1chunk.scaler.joblib"
)

RF_CKPT_PATH = Path(
    "/content/drive/MyDrive/saved_models/mission1/"
    "mission1_mfcc_rf.joblib"
)


# ============================================================
# 2. mission1 코드 import
# ============================================================

if str(MISSION_DIR) not in sys.path:
    sys.path.insert(0, str(MISSION_DIR))

if "common" in sys.modules:
    del sys.modules["common"]

import common

from common import (
    GENDER_TO_INDEX,
    load_annotation,
    load_caller_audio,
    waveform_to_mfcc_feature,
)

print("common.py:", common.__file__)


# ============================================================
# 3. Validation pair
#    train_mfcc_rf.py와 동일한 로직
# ============================================================

def collect_split_pairs(data_root: Path, split: str):

    if split not in {"Training", "Validation"}:
        raise ValueError(f"지원하지 않는 split: {split}")

    parts = sorted([
        p for p in data_root.iterdir()
        if p.is_dir()
    ])

    if split == "Training":
        audio_folder = "TS_서울_구급"
        label_folder = "TL_서울_구급"
    else:
        audio_folder = "VS_서울_구급"
        label_folder = "VL_서울_구급"

    audio_files = []
    label_files = []

    for part in parts:

        base = part / "대학부 데이터"

        audio_dir = (
            base
            / split
            / "1.원천데이터"
            / audio_folder
        )

        label_dir = (
            base
            / split
            / "2.라벨링데이터"
            / label_folder
        )

        if audio_dir.exists():
            audio_files.extend(
                audio_dir.rglob("*.wav")
            )

        if label_dir.exists():
            label_files.extend(
                label_dir.rglob("*.json")
            )

    label_map = {
        p.stem: p
        for p in label_files
    }

    pairs = []

    for audio_path in audio_files:

        json_path = label_map.get(
            audio_path.stem
        )

        if json_path is not None:
            pairs.append(
                (audio_path, json_path)
            )

    return sorted(
        pairs,
        key=lambda x: x[0].stem
    )


val_pairs = collect_split_pairs(
    DATA_ROOT,
    "Validation",
)

print(
    f"Validation pairs: {len(val_pairs):,}"
)

assert len(val_pairs) == 3640, (
    f"Validation pair 개수가 예상과 다름: "
    f"{len(val_pairs)}"
)


# ============================================================
# 4. WavLM 3-chunk cache
# ============================================================

cache = np.load(
    WAVLM_CACHE_PATH
)

X_wavlm = cache["x"].astype(
    np.float32
)

y_cache = cache["y"].astype(
    np.int64
)

print(
    "WavLM cache:",
    X_wavlm.shape,
    y_cache.shape,
)

assert X_wavlm.shape == (3640, 768)
assert y_cache.shape == (3640,)


# ============================================================
# 5. Validation label 순서 검증
# ============================================================

y_val = []

for _, json_path in val_pairs:

    annotation = load_annotation(
        json_path
    )

    gender = str(
        annotation.get(
            "gender",
            ""
        )
    ).strip().upper()

    if gender not in GENDER_TO_INDEX:
        raise ValueError(
            f"잘못된 gender: "
            f"{json_path} -> {gender}"
        )

    y_val.append(
        GENDER_TO_INDEX[gender]
    )


y_val = np.asarray(
    y_val,
    dtype=np.int64,
)


# ⭐ 매우 중요
# 3-chunk cache와 RF Validation 순서가 같은지 확인
if not np.array_equal(
    y_cache,
    y_val,
):
    raise RuntimeError(
        "❌ WavLM cache와 RF Validation label "
        "순서가 일치하지 않습니다.\n"
        "Fusion을 중단합니다."
    )

print(
    "✅ Validation label 순서 확인 완료"
)


# ============================================================
# 6. WavLM 3-chunk probability
# ============================================================

wavlm_scaler = joblib.load(
    WAVLM_SCALER_PATH
)

wavlm_ckpt = torch.load(
    WAVLM_HEAD_PATH,
    map_location="cpu",
    weights_only=False,
)


class WavLMHead(torch.nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        dropout,
    ):

        super().__init__()

        self.net = torch.nn.Sequential(
            torch.nn.Linear(
                input_dim,
                hidden_dim,
            ),
            torch.nn.ReLU(),
            torch.nn.Dropout(
                dropout,
            ),
            torch.nn.Linear(
                hidden_dim,
                2,
            ),
        )

    def forward(self, x):
        return self.net(x)


head = WavLMHead(
    int(
        wavlm_ckpt.get(
            "input_dim",
            768,
        )
    ),
    int(
        wavlm_ckpt.get(
            "hidden_dim",
            256,
        )
    ),
    float(
        wavlm_ckpt.get(
            "dropout",
            0.2,
        )
    ),
)

head.load_state_dict(
    wavlm_ckpt[
        "head_state_dict"
    ]
)

head.eval()


X_wavlm_scaled = (
    wavlm_scaler
    .transform(X_wavlm)
    .astype(np.float32)
)


with torch.inference_mode():

    wavlm_logits = head(
        torch.from_numpy(
            X_wavlm_scaled
        )
    )

    wavlm_prob = (
        torch.softmax(
            wavlm_logits,
            dim=1,
        )
        .numpy()
    )


wavlm_pred = wavlm_prob.argmax(
    axis=1
)

wavlm_acc = accuracy_score(
    y_val,
    wavlm_pred,
)

wavlm_f1 = f1_score(
    y_val,
    wavlm_pred,
    average="macro",
)

print("\nWavLM 3-Chunk")
print(
    f"Accuracy : {wavlm_acc:.4f}"
)
print(
    f"Macro F1 : {wavlm_f1:.4f}"
)


# ============================================================
# 7. 기존 RF checkpoint
# ============================================================

rf_ckpt = joblib.load(
    RF_CKPT_PATH
)

# train_mfcc_rf.py 저장 구조에 맞게 classifier 추출
rf_classifier = rf_ckpt[
    "classifier"
]

n_mfcc = int(
    rf_ckpt["n_mfcc"]
)

use_f0 = bool(
    rf_ckpt["use_f0"]
)

timestamp_unit = rf_ckpt.get(
    "timestamp_unit",
    "auto",
)

print("\nRF checkpoint")
print(
    "n_mfcc       :",
    n_mfcc,
)
print(
    "use_f0       :",
    use_f0,
)
print(
    "timestamp_unit:",
    timestamp_unit,
)
print(
    "stored val acc:",
    rf_ckpt.get(
        "validation_accuracy"
    ),
)


# ============================================================
# 8. RF Validation feature만 추출
# ============================================================

X_rf = []


print(
    "\nMFCC + F0 Validation feature 추출 시작"
)


for audio_path, json_path in tqdm(
    val_pairs,
    desc="RF Validation",
):

    annotation = load_annotation(
        json_path
    )

    waveform = load_caller_audio(
        audio_path,
        annotation,
        timestamp_unit=timestamp_unit,
    )

    feature = waveform_to_mfcc_feature(
        waveform,
        n_mfcc,
        use_f0=use_f0,
    )

    X_rf.append(
        feature
    )


X_rf = np.vstack(
    X_rf
).astype(
    np.float32
)

print(
    "RF feature shape:",
    X_rf.shape,
)


# ============================================================
# 9. RF probability
# ============================================================

rf_prob_raw = (
    rf_classifier.predict_proba(
        X_rf
    )
)


# classifier.classes_ 순서를
# WavLM의 class index와 맞춤
rf_prob = np.zeros(
    (
        len(X_rf),
        2,
    ),
    dtype=np.float32,
)


for col, class_id in enumerate(
    rf_classifier.classes_
):

    rf_prob[
        :,
        int(class_id)
    ] = rf_prob_raw[
        :,
        col
    ]


rf_pred = rf_prob.argmax(
    axis=1
)

rf_acc = accuracy_score(
    y_val,
    rf_pred,
)

rf_f1 = f1_score(
    y_val,
    rf_pred,
    average="macro",
)


print("\nMFCC + F0 RandomForest")
print(
    f"Accuracy : {rf_acc:.4f}"
)
print(
    f"Macro F1 : {rf_f1:.4f}"
)


# ============================================================
# 10. 안전 검증
#
# 기존 checkpoint에 저장된 Validation Accuracy와
# 지금 다시 계산한 RF Accuracy가 같은지 확인한다.
# ============================================================

stored_rf_acc = rf_ckpt.get(
    "validation_accuracy"
)

if stored_rf_acc is not None:

    difference = abs(
        float(stored_rf_acc)
        - rf_acc
    )

    print(
        "\nRF stored accuracy:",
        f"{float(stored_rf_acc):.4f}",
    )

    print(
        "RF current accuracy:",
        f"{rf_acc:.4f}",
    )

    if difference > 1e-8:

        raise RuntimeError(
            "❌ 기존 RF Validation Accuracy와 "
            "현재 재계산 결과가 다릅니다.\n"
            "Fusion 전에 전처리/순서를 확인해야 합니다."
        )


print(
    "✅ RF 결과 재현 확인 완료"
)


# ============================================================
# 11. WavLM 3-chunk + RF Late Fusion
# ============================================================

WAVLM_WEIGHT = 0.85
RF_WEIGHT = 0.15


fusion_prob = (
    WAVLM_WEIGHT
    * wavlm_prob
    +
    RF_WEIGHT
    * rf_prob
)

fusion_pred = fusion_prob.argmax(
    axis=1
)


fusion_acc = accuracy_score(
    y_val,
    fusion_pred,
)

fusion_f1 = f1_score(
    y_val,
    fusion_pred,
    average="macro",
)


# ============================================================
# 12. 최종 비교
# ============================================================

print("\n")
print("=" * 65)
print("MISSION 1 FINAL COMPARISON")
print("=" * 65)

print(
    f"MFCC + F0 RandomForest      "
    f"Accuracy={rf_acc:.4f}   "
    f"Macro-F1={rf_f1:.4f}"
)

print(
    f"WavLM 3-Chunk              "
    f"Accuracy={wavlm_acc:.4f}   "
    f"Macro-F1={wavlm_f1:.4f}"
)

print(
    f"WavLM 3-Chunk + RF Fusion  "
    f"Accuracy={fusion_acc:.4f}   "
    f"Macro-F1={fusion_f1:.4f}"
)

print("=" * 65)


# ============================================================
# 13. Fusion이 실제로 바꾼 예측 분석
# ============================================================

changed = (
    wavlm_pred
    != fusion_pred
)

fixed = (
    (wavlm_pred != y_val)
    &
    (fusion_pred == y_val)
)

broken = (
    (wavlm_pred == y_val)
    &
    (fusion_pred != y_val)
)


print(
    "\nFusion으로 예측 변경:",
    int(changed.sum()),
    "개",
)

print(
    "오답 → 정답:",
    int(fixed.sum()),
    "개",
)

print(
    "정답 → 오답:",
    int(broken.sum()),
    "개",
)

print(
    "순개선:",
    int(fixed.sum() - broken.sum()),
    "개",
)


print(
    "\nFusion classification report"
)

print(
    classification_report(
        y_val,
        fusion_pred,
        target_names=[
            "M",
            "F",
        ],
        digits=4,
        zero_division=0,
    )
)

common.py: /content/drive/MyDrive/mission1/common.py
Validation pairs: 3,640
WavLM cache: (3640, 768) (3640,)
✅ Validation label 순서 확인 완료

WavLM 3-Chunk
Accuracy : 0.9750
Macro F1 : 0.9748

RF checkpoint
n_mfcc       : 20
use_f0       : True
timestamp_unit: auto
stored val acc: 0.9423076923076923

MFCC + F0 Validation feature 추출 시작


RF Validation:   0%|          | 0/3640 [00:00<?, ?it/s]

RF feature shape: (3640, 246)

MFCC + F0 RandomForest
Accuracy : 0.9423
Macro F1 : 0.9419

RF stored accuracy: 0.9423
RF current accuracy: 0.9423
✅ RF 결과 재현 확인 완료


MISSION 1 FINAL COMPARISON
MFCC + F0 RandomForest      Accuracy=0.9423   Macro-F1=0.9419
WavLM 3-Chunk              Accuracy=0.9750   Macro-F1=0.9748
WavLM 3-Chunk + RF Fusion  Accuracy=0.9755   Macro-F1=0.9754

Fusion으로 예측 변경: 12 개
오답 → 정답: 7 개
정답 → 오답: 5 개
순개선: 2 개

Fusion classification report
              precision    recall  f1-score   support

           M     0.9789    0.9679    0.9734      1681
           F     0.9727    0.9821    0.9774      1959

    accuracy                         0.9755      3640
   macro avg     0.9758    0.9750    0.9754      3640
weighted avg     0.9756    0.9755    0.9755      3640



## 9. 최종 실험 결과

| Model | Accuracy | Macro F1 |
| --- | ---: | ---: |
| Log-Mel CNN | 0.9396 | 0.9393 |
| MFCC without F0 + RF | 0.9365 | 0.9361 |
| MFCC + F0 + RF | 0.9423 | 0.9419 |
| WavLM 1-Chunk | 0.9566 | 0.9563 |
| WavLM 1-Chunk + RF Fusion | 0.9635 | 0.9632 |
| WavLM 3-Chunk | 0.9750 | 0.9748 |
| WavLM 3-Chunk + RF Fusion | 0.9755 | 0.9754 |

## 10. 최종 모델 선정

최종 제출 모델은 WavLM-base-plus + 3-Chunk inference이다. 1-chunk 실험에서 학습한 동일한 scaler와 head에 3-chunk inference를 적용했을 때 Validation Accuracy가 0.9566에서 0.9750으로 1.84%p 높아졌다.

Late fusion은 Validation에서 0.9755를 기록했지만, 3-chunk 단독 대비 차이는 0.05%p이고 예측의 순개선도 2개였다. 또한 fusion weight는 동일한 Validation set에서 선택됐으며, 적용하려면 MFCC/F0 feature extraction과 RandomForest가 추가로 필요하다. 따라서 fusion은 추가 실험 결과로 보존하고, 제출 inference는 더 단순한 WavLM-base-plus + 3-Chunk inference 단일 모델로 선정했다.
